# Seizure detection from EEG data

---

# Preview into dataset

EEG signals have been collected from recordings from 10 different subjects, in total there are 739 EEG recordings (i.e. files). The number of recordings between subjects varies, but each recording is 10 seconds long time window. Recordings are further devided into "seizure" and "nonseizure" subsets. "seizure" subset has 339 EEG signals with seizures. Similarly "non-seizure" subset has 400 records of normal EEG signal. Each record is saved as CSV and has 23 columns that correspond to 23 unique EEG channels. The filename includes the event number and the subject number. For example, 0 is the event number, and 01 is the subject number in “sample_chb01_0.csv.” Note that the subjects have both the “nonseizure” and “seizure” events.

### Overview
- **Total recordings:** 739
- **Subjects:** 10
- **Recording duration:** 10 seconds each
- **File format:** CSV with 23 columns (EEG channels)

## Subsets
| Subset | Count | Description |
|---|---|---|
| Seizure | 339 | EEG recordings with seizure events |
| Non-seizure | 400 | Normal EEG recordings |
| **Total** | **739** | |

### Hierarchy
```
EEG Dataset (739 recordings · 10 subjects)
│
├── Subject 01
│   ├── Seizure recordings
│   └── Non-seizure recordings
│
├── Subject 02
│   ├── Seizure recordings
│   └── Non-seizure recordings
│
├── · · ·
│
└── Subject 10
    ├── Seizure recordings
    └── Non-seizure recordings
```

### Recording (CSV File)
- **Duration:** 10-second time window
- **Columns:** 23 (one per EEG channel)
- **Filename format:** `sample_chb[subj]_[event].csv`
- **Example:** `sample_chb01_0.csv` → subject `01`, event `0`

<div>
    <img src="seizure-electrode-positioning.png">
</div>

---

# Setup 

In [ ]:
import os
from collections import defaultdict
import pandas as pd
import numpy as np 
import glob
from pprint import pp

import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
seizure = defaultdict(list)
nonseizure = defaultdict(list)

seizure_data = glob.glob("data/seizure/*.csv")
nonseizure_data = glob.glob("data/nonseizure/*.csv")


In [ ]:
def getdata(path, dictionary):
    for file in path:
        df = pd.read_csv(file)
        dictionary[file].append(df)

getdata(seizure_data, seizure)
getdata(nonseizure_data, nonseizure)

seizure_keys = seizure.keys()
nonseizure_keys = nonseizure.keys()
display(seizure)

In [ ]:
for i, path in enumerate(nonseizure.keys()):
    print(f"{i+1}. {path}")

# Visualize data

In [ ]:
# Extract patient numbers from file paths
def get_patient_number(filepath):
    """Extract patient number from filepath like 'data/seizure/sample_chb01_0.csv'"""
    filename = filepath.split('/')[-1]  # Get 'sample_chb01_0.csv'
    patient_num = filename.split('chb')[1][:2]  # Extract '01'
    return patient_num


def plot_overview(seizure_data, nonseizure_data, num_patients=3):
    # Get unique patients
    seizure_patients = {}
    nonseizure_patients = {}

    for path in seizure_data.keys():
        patient = get_patient_number(path)
        if patient not in seizure_patients:
            seizure_patients[patient] = path

    for path in nonseizure_data.keys():
        patient = get_patient_number(path)
        if patient not in nonseizure_patients:
            nonseizure_patients[patient] = path

    # Select first num_patients patients
    selected_patients = sorted(seizure_patients.keys())[:num_patients]

    # Create subplots: 2 rows (seizure and non-seizure), num_patients columns
    fig, axes = plt.subplots(2, num_patients, figsize=(5*num_patients, 10))

    # Plot seizure signals in top row
    for idx, patient in enumerate(selected_patients):
        ax = axes[0, idx]
        path = seizure_patients[patient]
        df = seizure_data[path][0]
        
        # Plot all 23 channels
        for channel in df.columns:
            ax.plot(df[channel], alpha=0.7, linewidth=0.8)
        
        ax.set_title(f'Seizure - Patient {patient}', fontsize=12, fontweight='bold')
        ax.set_xlabel('Sample')
        ax.set_ylabel('Amplitude (μV)')
        ax.grid(True, alpha=0.3)

    # Plot non-seizure signals in bottom row
    for idx, patient in enumerate(selected_patients):
        ax = axes[1, idx]
        path = nonseizure_patients[patient]
        df = nonseizure_data[path][0]
        
        # Plot all 23 channels
        for channel in df.columns:
            ax.plot(df[channel], alpha=0.7, linewidth=0.8)
        
        ax.set_title(f'Non-Seizure - Patient {patient}', fontsize=12, fontweight='bold')
        ax.set_xlabel('Sample')
        ax.set_ylabel('Amplitude (μV)')
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()


# Call the function with default num_patients=3
plot_overview(seizure, nonseizure, num_patients=3)


def plot_patient_channels(seizure_data, nonseizure_data, patient_num):
    seizure_patients = {}
    nonseizure_patients = {}

    for path in seizure_data.keys():
        patient = get_patient_number(path)
        if patient not in seizure_patients:
            seizure_patients[patient] = path

    for path in nonseizure_data.keys():
        patient = get_patient_number(path)
        if patient not in nonseizure_patients:
            nonseizure_patients[patient] = path
    
    seizure_path = seizure_patients[patient_num]
    nonseizure_path = nonseizure_patients[patient_num]
    seizure_df = seizure_data[seizure_path][0]
    nonseizure_df = nonseizure_data[nonseizure_path][0]

    # Plot channels
    fig, axes = plt.subplots(6, 4, figsize=(18, 12), layout='constrained')
    axes = axes.flatten()

    fig.suptitle(f"Channels - Patient {patient_num}", fontsize=16, fontweight='bold')
    fig.supxlabel('Sample', fontsize=12, fontweight='bold')
    fig.supylabel('Amplitude (μV)', fontsize=12, fontweight='bold')



    # Plot all 23 channels
    for i, (seiz_ch, nonseiz_ch) in enumerate(zip(seizure_df.columns, nonseizure_df.columns)):
        axes[i].plot(nonseizure_df[nonseiz_ch], alpha=0.7, linewidth=0.8, color='green')
        axes[i].plot(seizure_df[seiz_ch], alpha=0.7, linewidth=0.8, color='red')

        axes[i].set_title(f'Ch: {seiz_ch}', fontsize=12)
        axes[i].grid(True, alpha=0.3)

    # remove empty subplot
    fig.delaxes(axes[-1])


    plt.tight_layout()
    plt.show()

plot_patient_channels(seizure, nonseizure, '02')

# Preprocessing

1) Do Discrete Wavelet Transform for EEG channels so that we can create signal vector that we will later on feed to the model.

In [ ]:
import pywt
from scipy.stats import zscore

# Do wavelet transform for every patients every channel
# Concatenate these together to create 1d array

def process_segment_2d(segment_df, wavelet='db1', level=3):
    channel_features = []
    for col in segment_df.columns:
        coeffs = pywt.wavedec(segment_df[col].values, wavelet, level=level)
        normalized_channels = zscore(np.concatenate(coeffs))
        channel_features.append(normalized_channels)
    return np.stack(channel_features, axis=-1)  # (dwt_len, n_channels)


def aggregate_patient_signals(seizure_data, non_seizure_data):
    """
    return wavelet transformed signals in dict:

    signals: {
        patient_1: 
            seizure: [s1, s2, s3, ..., sn]
            non-seizure: [s1, s2, s3, ..., sn]
        patient_2:
            seizure: [s1, s2, s3, ..., sn]
            non-seizure: [s1, s2, s3, ..., sn]
        patient_3:
            seizure: [s1, s2, s3, ..., sn]
            non-seizure: [s1, s2, s3, ..., sn]
        ...
        patient_n:
            seizure: [s1, s2, s3, ..., sn]
            non-seizure: [s1, s2, s3, ..., sn]
    }
    """

    patients = []

    # initiate patients list
    for path in seizure_data:
        patient_id = get_patient_number(path) # '01'
        if patient_id not in patients:
            patients.append(patient_id)

    patients.sort()

    signals = {p: {'seizure': [], 'non-seizure': []} for p in patients}

    # append data for patient
    for path in seizure_data:
        patient_id = get_patient_number(path)
        signal_df = seizure_data[path][0] # unwrap df from list

        signals[patient_id]['seizure'].append(signal_df)

    for path in non_seizure_data:
        patient_id = get_patient_number(path)
        signal_df = non_seizure_data[path][0] # unwrap df from list

        signals[patient_id]['non-seizure'].append(signal_df)

    for k, v in signals.items():
        print(f"{k}:\n\tseizure: {len(v['seizure'])}\n\tnon-seizure: {len(v['non-seizure'])}")

    display(signals['01']['seizure'][0])

    return signals


patient_signals = aggregate_patient_signals(seizure, nonseizure)


X_train, X_test, y_train, y_test = [], [], [], []

for patient_id, segments in patient_signals.items():
    seizure_segments = segments['seizure']
    non_seizure_segments = segments['non-seizure']

    n_sz_train = int(0.7 * len(seizure_segments))
    n_ns_train = int(0.7 * len(non_seizure_segments))

    for seg in seizure_segments[:n_sz_train]:
        X_train.append(process_segment_2d(seg))
        y_train.append(1)
    for seg in seizure_segments[n_sz_train:]:
        X_test.append(process_segment_2d(seg))
        y_test.append(1)
    for seg in non_seizure_segments[:n_ns_train]:
        X_train.append(process_segment_2d(seg))
        y_train.append(0)
    for seg in non_seizure_segments[n_ns_train:]:
        X_test.append(process_segment_2d(seg))
        y_test.append(0)

X_train = np.array(X_train)  # (512, 2560, n_channels)
X_test  = np.array(X_test)
y_train = np.array(y_train)
y_test  = np.array(y_test)

print(f"Mean: {X_train[0].mean()}")
print(f"Std: {X_train[0].std()}")

print(X_train.shape)  # expect (512, 2560, n_channels)


In [ ]:
# Build model
import keras
from keras import layers, models, optimizers
from keras.regularizers import L2

# Add this to detect the issue:
print(f"X_train has NaN: {np.isnan(X_train).any()}")
print(f"X_train has Inf: {np.isinf(X_train).any()}")
print(f"X_train min: {X_train.min()}, max: {X_train.max()}")

def build_model(input_shape):
    model = models.Sequential()

    model.add(layers.Conv1D(16, 4, strides=4, padding='valid', input_shape=input_shape))
    model.add(layers.BatchNormalization())
    model.add(layers.Activation('relu'))

    model.add(layers.Conv1D(32, 4, strides=2, padding='valid'))
    model.add(layers.BatchNormalization())
    model.add(layers.Activation('relu'))
    model.add(layers.MaxPooling1D(3, 2))

    model.add(layers.Conv1D(64, 3, strides=2, padding='valid'))
    model.add(layers.BatchNormalization())
    model.add(layers.Activation('relu'))
    model.add(layers.MaxPooling1D(3, 2))

    model.add(layers.LSTM(32, dropout=0.3, recurrent_dropout=0.2))

    model.add(layers.Dense(32, activation='relu', kernel_regularizer=L2(0.05)))
    model.add(layers.Dropout(0.5))
    model.add(layers.Dense(1, activation='sigmoid'))

    model.compile(
        optimizer=optimizers.Adam(learning_rate=0.0001),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

    return model

model = build_model((X_train.shape[1], X_train.shape[2]))
model.summary()

In [ ]:
history = model.fit(
    X_train, y_train,
    epochs=300,
    batch_size=40,
    validation_data=(X_test, y_test),
    verbose=1
)

val_acc = max(history.history['val_accuracy'])
print(f"  → val_accuracy: {val_acc:.4f}")

In [ ]:
from keras import backend as K

# Build full dataset keeping patient labels
X_by_patient = {}
y_by_patient = {}

for patient_id, segments in patient_signals.items():
    X_patient = []
    y_patient = []
    for seg in segments['seizure']:
        X_patient.append(process_segment_2d(seg))
        y_patient.append(1)
    for seg in segments['non-seizure']:
        X_patient.append(process_segment_2d(seg))
        y_patient.append(0)
    X_by_patient[patient_id] = np.array(X_patient)
    y_by_patient[patient_id] = np.array(y_patient)

patient_ids = list(X_by_patient.keys())
fold_results = []

for i, test_patient in enumerate(patient_ids):
    print(f"\nFold {i+1}/{len(patient_ids)} — test patient: {test_patient}")

    # Build train set from all other patients
    train_patients = [p for p in patient_ids if p != test_patient]
    X_train = np.concatenate([X_by_patient[p] for p in train_patients])
    y_train = np.concatenate([y_by_patient[p] for p in train_patients])
    X_test  = X_by_patient[test_patient]
    y_test  = y_by_patient[test_patient]

    print(f"  Train: {X_train.shape}, Test: {X_test.shape}")

    # Reset model each fold
    K.clear_session()
    model = build_model((X_train.shape[1], X_train.shape[2]))

    history = model.fit(
        X_train, y_train,
        epochs=300,
        batch_size=32,
        validation_data=(X_test, y_test),
        verbose=0
    )

    best_val_acc = max(history.history['val_accuracy'])
    fold_results.append({'patient': test_patient, 'val_accuracy': best_val_acc})
    print(f"  → val_accuracy: {best_val_acc:.4f}")

# Summary
print("\nLOPO Results")
for r in fold_results:
    print(f"  {r['patient']}: {r['val_accuracy']:.4f}")
accs = [r['val_accuracy'] for r in fold_results]
print(f"\nMean: {np.mean(accs):.4f} ± {np.std(accs):.4f}")